# ????????????   ??????



????? ?????? ?????? ???????? ?????????? ?????? **????????????? ?????????????**: ?? ???? ?????? ????? ????????? ?????? ? ????????, ? ?? ?????? ??? ?????? ????????????? ???????? ??????? ?????? ???????? (?????? ??? ?????? 50000$ ?? ???).

## ???????? ?????????

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#?????????? ???????, ?????????, ??? ??????? ????? ??????? ????? ? ????????: 
%matplotlib inline 

In [ ]:
#!pip install comet_ml

In [ ]:
from comet_ml import Experiment

In [ ]:
#@title CometML set up
project_name = 'predict-income'
workspace = 'ketrint'
api_key = ''

In [ ]:
experiment = Experiment(project_name=project_name, workspace=workspace, api_key = api_key, log_code=True)

## ???????? ??????

In [ ]:
%%capture
!wget https://www.dropbox.com/s/1hfrcm09qtvoeoz/train.csv
!wget https://www.dropbox.com/s/jj7c8r0x57zda0a/test.csv

In [ ]:
training_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

In [ ]:
training_data.head()

???????? ??????????? ????? ??????:

In [ ]:
training_data.shape, test_data.shape

**???????? ? ??????? ??????????**



In [ ]:
target_name = 'income'

In [ ]:
y_train = training_data[target_name]
y_test = test_data[target_name]

X_train = training_data.drop(columns=[target_name])
X_test = test_data.drop(columns=[target_name])

**?????? ?????????????????? ???????**

In [ ]:
y_train.value_counts(normalize=True)

????? ???????, ????? ?????? **?????????????????? ???????? ?????????????**. 

## ????? ?????????? ?? ???????? ??????



In [ ]:
X_train.describe().T

In [ ]:
X_train.hist(figsize=(12,8))
experiment.log_figure('X_train_numeric_distr', plt)

??? ?? ????? ??????? ?? ????????????
* ???????? ?????????? ? ???????? ?????? ?????? 4;
* ??????? ???????????? ??????????? ? ????????? ?? 17 ?? 90 ??? ? ? ??????? ?????????? 39 ???;
* ????? ???????? ? ????????? capital-gain ? capital-loss (???????? count ?????? ??????? ??????). ?? ??????????? ????? ????????, ??? ??????? ?????????? ???????? ? ???? ????????? - ??? ?????? ????;

## ????????????? ??????

### ???????? ?? ??????? ????????? ? ????? ??????????



In [ ]:
X_train.info()

???? ? ????? ??????:
* 8 ????????? ???? **object** - ?????? ??? ????????? ????????;
* 3 ???????? ???? **int64** - ????????????? ????????;
* 2 ???????? ???? **float64** - ????? ? ????????? ???????. 

????? ?????, ?? ?????, ??? ???????? *workclass, education, capital-gain ? capital-loss* ????? ???????? ? ??????.

### ????????? ????????? ? ??????

?????? ?? ??? ????????? ????? ???????? ? ?????????? ? ??????. ????? ?????, ?????? ???????? ????????, ?? ?????? ????? ???????????? ????????????? ??????, ? ?????? ????? ????????? ???????. 

????? ?????????? ?????? ??? ????????? ?????????:
- ???????? ???????? ? ?????????? (?? ?????????? ??? ??????? ?????????? ?????????, ?????? ??? ???????? ??????);
- ?????????? ???????/????????? ?????????;
- ??????????? ????? ?????? ?????????;
- ?????????? ????????? ?????????.

**?????:** ???????? ??????? ?? ?????? ???????????? ?????, ??? ? ????????? ???????. 

In [ ]:
X_train['capital-loss'] = X_train['capital-loss'].fillna(0)
X_train['capital-gain'] = X_train['capital-gain'].fillna(0)

??????? ?? ?? ????? ??? ???????? ???????:

In [ ]:
X_test['capital-loss'] = X_test['capital-loss'].fillna(0)
X_test['capital-gain'] = X_test['capital-gain'].fillna(0)

In [ ]:
X_train['workclass'].value_counts(True).head(1)

In [ ]:
X_train['education'].value_counts(True).head(1)

???????? ?????? ???????? ???????? **workclass** - Private, ???????? **education** - HS-grad. 

?????? ???? ???????? ????????:

In [ ]:
X_train['workclass'] = X_train['workclass'].fillna('Private')
X_train['education'] = X_train['education'].fillna('HS-grad')

??????? ?? ?? ????? ??? ???????? ???????:

In [ ]:
X_test['workclass'] = X_test['workclass'].fillna('Private')
X_test['education'] = X_test['education'].fillna('HS-grad')

### ????????? ?????????????? ??????????

? ?????? 8 ???????? ???????? ??????????????? ??????????. ??? ??? ? ????? ?????? ??? ???????? **?????????**. ?????? ?????? ???????????? ????????? ????????, ??????? ??? ?????????? ???????????? ??, ?? ???? ????????????? ? ????????. 

??? ?????? ?????????? ???????? ????????????? ????????:

In [ ]:
categorical_features = ['workclass', 'education', 'marital-status', 
                        'occupation', 'relationship', 'race', 'sex', 
                        'native-country']

non_categorical_features = ['age', 'capital-gain', 'capital-loss', 
                            'hours-per-week']

????????? ?? ??????? ?????????????? ??????????:

In [ ]:
X_train[categorical_features].head()

?? ?????????? ??? ?????????? ?????? ??????????? ?????????????? ?????????:
* Label Encoding, ???????? ????????????? ????? `LabelEncoder()` ?? ?????????? **sklearn**
* One Hot Encoding, ???????? ????????????? ????? `OneHotEncoder()` ?? ?????????? **sklearn**

**Label Encoding**



In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
label_encoder = LabelEncoder()

In [ ]:
X_train_le = X_train.copy()

In [ ]:
X_test_le = X_test.copy()

In [ ]:
for feature in categorical_features:
    X_train_le[feature] = label_encoder.fit_transform(X_train_le[feature])
    X_test_le[feature] = label_encoder.transform(X_test_le[feature])

In [ ]:
X_train_le.head()

**One Hot Encoding**



In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
onehot_encoder = OneHotEncoder()

In [ ]:
# '????' ? ????? ????????? ?????????????? ? ???????, ????????? ????????? ? ??????
X_train_onehot = onehot_encoder.fit_transform(X_train[categorical_features]).toarray()

# ????????? ?????????? ?????????????? ? ???????? ???????, ????????? ????????? ? ??????
X_test_onehot = onehot_encoder.transform(X_test[categorical_features]).toarray()

# ??????? ?????????? ???????? ????? ??????? ? ????????? ??????????
columns = onehot_encoder.get_feature_names(categorical_features)

In [ ]:
X_train_onehot = pd.DataFrame(X_train_onehot, columns=columns)
X_test_onehot = pd.DataFrame(X_test_onehot, columns=columns)

? ???????, ??????? ? ????? ?????????? ?????????? ?????????, ?????? ??? ???????????????? ?????????????? ???????? ?? ????? ???????. 

In [ ]:
# ????????? ????? ???????? ? ????????
X_train_new = pd.concat([X_train, X_train_onehot], axis=1)
X_test_new = pd.concat([X_test, X_test_onehot], axis=1)

# ??????? ?????? ?????????????? ????????
X_train_new = X_train_new.drop(columns=categorical_features)
X_test_new = X_test_new.drop(columns=categorical_features)

In [ ]:
X_train_new.head()

## ???????? ?????? 

## ????? ?????? ???????? ??????????????

??? ???????? ?????????????? ????????????? ????????????? ??????????.
 

In [ ]:
from sklearn.linear_model import LogisticRegression

??????? ???????? ?????? ??????:

In [ ]:
linear_model = 

## ???????? ?????? ?? ?????? ????? ?????????????? ??????

In [ ]:
linear_model_oh.fit(X_train_new, y_train)

In [ ]:
linear_model.fit(X_train_le, y_train)

# ????????? ?????? ?? ???????? ???????

In [ ]:
prediction_oh = linear_model_oh.predict(X_test_new)

In [ ]:
np.unique(prediction_oh)

In [ ]:
# ??????? 1?? ?????? ??????????? ? ??????????????

In [ ]:
# ??????? 1?? ?????? ??????????? ? y_test?

## ???????? ????????????

????????? ???????? (**accuracy**) ?????? ???????. ????????, ??? ???????? ????????, ??????? ? ?????????? ??????????? ?? ??????? ???????? ??????? ?????? ??????? 1 ? ??????? ??????? 0.

??????? ??? ???????? ???????? ??????????? ? ?????????? **sklearn** ? ?????????? `accuracy_score()`.

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
acc_oh = accuracy_score(y_test, prediction_oh)

In [ ]:
experiment.log_metric('Accuracy OneHot data', acc_oh)

??????? ???????? ???????? ??? ????? ??????? ? ????????? ??? ???????????? ??????????????, ??????, ??????? ?????? ?? ????????????? ??????? ?????, ? ????? ?????? 0. ??? ????? ????? ? ??????? `accuracy_score()` ? ???????? ??????? ????????? ???????? ?????? ????? ?????? ?? ???????. ??? ???????? ? ??????? ??????? `zeros_like()` ?? ?????????? **numpy**.

In [ ]:
print('Accuracy Major Class: ', accuracy_score(y_test, np.zeros_like(y_test)).round(3))

In [ ]:
# ???????????? accuracy ??? LE

## ??????? ????????????? ?????? ?????????????

?????? ?????? ????????? ???????? ?????? ?????????????? - ???????????? ??????? ?????????????. 


|                   |  ??????????? `0` |  ??????????? `1` |
|:-------------------|:------------------|:------------------|
|**??????????** `0`  |       TN         |       FP         |
|**??????????** `1`  |       FN         |       TP         | 


In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
pd.DataFrame(confusion_matrix(y_test, prediction_oh))

In [ ]:
experiment.log_confusion_matrix(y_test, prediction_oh)

## ??????? ??????????? ?????????????? ? ??????

In [ ]:
pred_proba_oh = linear_model_oh.predict_proba(X_test_new)

In [ ]:
# ??? ???????? ?????????????

In [ ]:
pred_proba = pred_proba_oh[:, 1]

????????? ?? ????????????? ????????????? ??????????????? ??????????? ?????? > 50?, ????? `1`.

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(pred_proba, bins=100)
plt.vlines(0.5, 0, 2500)

plt.xlabel('????????????? ??????????? ??????', fontsize=20)
plt.ylabel('?????????? ?????', fontsize=20)

experiment.log_figure('predicted_income_probability', plt)

##ROC ?????? ??????????????

???? ????? ???????? ??????? ?? ?????? ??????? ??????, ?????? ???????? ?? ? ??????????? ?????????? True Positive ? False Positive, ? ? ?? ??????:

* ???? ?????????????????? ???????????? $\text{FPR} = \frac{FP}{FP + TN}$;
* ???? ??????? ????????????? ???????????? $\text{TPR} = \frac{TP}{TP + FN}$.

???????, ??? $FP + TN$ ???? ????? ????? ???????? ?????? $0$, ? $TP + FN$ - ????? ????? ???????? ?????? $1$. 

????? ?? ????? ?????????? ?????? ??? ?????? ????????????? ???????? ROC ??????. ROC ???????????????? ??? *Receiver Operating Characteristic*. ??? ?????? ???????? ?????????? ??????????? ???? ??????? ?????????? ???????????? (**TPR**) ?? ???? ????? ?????????? ???????????? (**FPR**) ??? ????????? ?????? ?????????????.

??????? `roc_curve()` ?? **scikit-learn** ????????? ???????? ?????????? ????? ROC ??????, ? ????? ???????? ?????? `threshold`, ??? ??????? ??????????? ??????????????? ???????? ?????? **FPR** ? **TPR**.

?? ???? ??????? `roc_curve()` ?????????? ???????? ??? ?????????:
* ???????? ???????? ?????
* ???????????, ????????????? ???????

In [ ]:
from sklearn.metrics import roc_curve

In [ ]:
false_positive_rates, true_positive_rates, threshold = roc_curve(y_test, pred_proba)

In [ ]:
# ??????? ??????
plt.figure(figsize=(7, 7))

# ?????? ??????
plt.plot(false_positive_rates, true_positive_rates, 
         label='?????????? ???????? ROC-AUC')

# ??????, ??????????????? ?????????? ??????????
plt.plot([0, 1], [0, 1], color='k', lw=2, linestyle=':', 
         label='??????, ???????? ????????? ????????')

plt.title('ROC curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.fill_between(false_positive_rates, true_positive_rates, step="pre", 
                 alpha=0.4, label='??????? ??? ?????? (ROC-AUC)')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
roc_auc = roc_auc_score(y_test, pred_proba).round(3)

In [ ]:
print("ROC-AUC ?? ???????? ???????:", roc_auc_score(y_test, pred_proba).round(3)) 

In [ ]:
experiment.log_metric('roc_auc OneHot data', roc_auc)

In [ ]:
pred_proba_le = linear_model.predict_proba(X_test_new)

In [ ]:
pred_proba = pred_proba_le[:, 1]

In [ ]:
roc_auc = roc_auc_score(y_test, pred_proba).round(3)
roc_auc